In [12]:
import numpy as np

# -----------------------------------------
# STEP 1: Define the Hidden States (Tags)
# -----------------------------------------
tags = ["Noun", "Verb", "Adverb", "Adjective"]

# -----------------------------------------
# STEP 2: Define the Vocabulary
# -----------------------------------------
words = ["Maha", "is", "not", "feeling", "well"]

# Convert words into indices
word_to_index = {
    "Maha": 0,
    "is": 1,
    "not": 2,
    "feeling": 3,
    "well": 4
}

# -----------------------------------------
# STEP 3: Initial Probability
# P(Tag at first word)
# -----------------------------------------
init_prob = np.array([
    0.70,  # Noun
    0.10,  # Verb
    0.10,  # Adverb
    0.10   # Adjective
])

# -----------------------------------------
# STEP 4: Transition Probability Matrix
# P(Current Tag | Previous Tag)
#
#              To
#         Noun  Verb  Adverb  Adjective
# From Noun
#      Verb
#      Adverb
#      Adjective
# -----------------------------------------
transition = np.array([
    [0.10, 0.70, 0.10, 0.10],  # From Noun
    [0.10, 0.10, 0.50, 0.30],  # From Verb
    [0.05, 0.70, 0.05, 0.20],  # From Adverb
    [0.25, 0.25, 0.25, 0.25]   # From Adjective
])

# -----------------------------------------
# STEP 5: Emission Probability Matrix
# P(Word | Tag)
#
#             Maha   is    not   feeling  well
# Noun        0.60  0.05  0.05   0.10    0.20
# Verb        0.05  0.50  0.05   0.35    0.05
# Adverb      0.02  0.03  0.85   0.05    0.05
# Adjective   0.05  0.05  0.05   0.05    0.80
# -----------------------------------------
emission = np.array([
    [0.60, 0.05, 0.05, 0.10, 0.20],  # Noun
    [0.05, 0.50, 0.05, 0.35, 0.05],  # Verb
    [0.02, 0.03, 0.85, 0.05, 0.05],  # Adverb
    [0.05, 0.05, 0.05, 0.05, 0.80]   # Adjective
])

# -----------------------------------------
# STEP 6: Convert probabilities into Log Space
# This avoids numerical underflow.
# -----------------------------------------
log_init = np.log(init_prob + 1e-10)
log_trans = np.log(transition + 1e-10)
log_emit = np.log(emission + 1e-10)

# -----------------------------------------
# STEP 7: Viterbi Algorithm
# -----------------------------------------
def viterbi(sentence):
    # Number of words
    T = len(sentence)
    # Number of tags
    N = len(tags)

    # DP table
    dp = np.full((N, T), -np.inf)

    # Backpointer table
    backpointer = np.zeros((N, T), dtype=int)

    # ---------------- Initialization ----------------
    # Calculate probability for the first word
    dp[:, 0] = log_init + log_emit[:, sentence[0]]

    # ---------------- Recursion ----------------
    # Process remaining words
    for t in range(1, T):
        # Check every current tag
        for j in range(N):
            # Calculate score from every previous tag
            scores = (
                dp[:, t-1]
                + log_trans[:, j]
                + log_emit[j, sentence[t]]
            )
            # Store best previous tag
            backpointer[j, t] = np.argmax(scores)
            # Store best score
            dp[j, t] = np.max(scores)

    # ---------------- Backtracking ----------------
    # Best tag for last word
    best_path = [np.argmax(dp[:, -1])]

    # Trace backwards
    for t in range(T-1, 0, -1):
        best_path.append(backpointer[best_path[-1], t])

    # Reverse to get correct order
    best_path.reverse()

    return best_path

# -----------------------------------------
# STEP 8: Example Sentence
# -----------------------------------------
sentence = ["Maha", "is", "not", "feeling", "well"]

# Convert words into indices
sentence_index = [word_to_index[word] for word in sentence]

# Run Viterbi
best_tags = viterbi(sentence_index)

# -----------------------------------------
# STEP 9: Print Results
# -----------------------------------------
print("Sentence:", sentence)
print("\nPredicted Tags:")
for word, tag in zip(sentence, best_tags):
    print(word, "->", tags[tag])

Sentence: ['Maha', 'is', 'not', 'feeling', 'well']

Predicted Tags:
Maha -> Noun
is -> Verb
not -> Adverb
feeling -> Verb
well -> Adjective
